In [0]:
!pip install python-dotenv

In [0]:
import sys
import os
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, avg, stddev, col, when
from dotenv import load_dotenv

load_dotenv()

In [0]:
# 3. Credentials
storage_account_name = "coinmarketcapapi"
container_name = "coin-market-cap-api-data"
storage_account_key = os.getenv("AZURE_STORAGE_KEY")

# Set the key on the DFS endpoint (Required for HNS)
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

print(f"🚀 Spark {spark.version} Ready for Gold Layer Transformations!")

In [0]:
storage_account_name = "coinmarketcapapi"
container_name = "coin-market-cap-api-data"
storage_account_key = os.getenv("AZURE_STORAGE_KEY")

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

# --- 2. PATH DEFINITIONS ---
silver_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver"
gold_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/gold"

silver_quotes_path = f"{silver_base_path}/crypto_quotes"
silver_global_path = f"{silver_base_path}/global_metrics"

gold_daily_summary_path = f"{gold_base_path}/daily_price_summary"
gold_volatility_path = f"{gold_base_path}/volatility_metrics"

In [0]:
# --- 2. DAILY PRICE SUMMARY & MOMENTUM ---
# Creating a table for day-to-day trends and price movements
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, avg, stddev, col

# --- 1. LOAD SILVER DATA ---
quotes_df = spark.read.format("delta").load(silver_quotes_path)
global_df = spark.read.format("delta").load(silver_global_path)

# --- 2. WINDOW SPECIFICATION ---
# Partition by coin and order by time to calculate rolling metrics
window_spec = Window.partitionBy("symbol").orderBy("event_timestamp")

# --- 3. DAILY PRICE SUMMARY & MOMENTUM ---
gold_daily_summary = quotes_df \
    .withColumn("prev_day_price", lag("price_usd", 1).over(window_spec)) \
    .withColumn("daily_return", 
        when(col("prev_day_price").isNull(), 0)
        .otherwise((col("price_usd") - col("prev_day_price")) / col("prev_day_price"))) \
    .withColumn("7d_moving_avg", avg("price_usd").over(window_spec.rowsBetween(-7, 0))) \
    .withColumn("14d_moving_avg", avg("price_usd").over(window_spec.rowsBetween(-14, 0))) \
    .withColumn("7d_avg_volume", avg("volume_24h").over(window_spec.rowsBetween(-7, 0))) \
    .withColumn("volume_spike", 
        when(col("volume_24h") > (col("7d_avg_volume") * 2), 1).otherwise(0))

# 4. Integrate Market Dominance
# We join with global metrics to see how much of the total market this specific coin represents
latest_global = global_df.select("event_timestamp", "total_market_cap")

gold_daily_summary = gold_daily_summary.join(latest_global, "event_timestamp", "left") \
    .withColumn("market_dominance_pct", (col("market_cap") / col("total_market_cap")) * 100)

gold_volatility_metrics = quotes_df \
    .withColumn("rolling_std", stddev("price_usd").over(window_spec.rowsBetween(-14, 0))) \
    .select("symbol", "rolling_std", "market_cap", "event_timestamp")

# --- 5. WRITE TO GOLD LAYER ---
gold_daily_summary.write.format("delta").mode("append").save(gold_daily_summary_path)
gold_volatility_metrics.write.format("delta").mode("append").save(gold_volatility_path)

print("✅ Gold Layer processing complete and saved to ADLS Gen2!")

In [0]:
# --- 1. VALIDATE DAILY SUMMARY & MOMENTUM ---
print("📊 Checking Daily Summary & 7-Day Moving Averages:")
gold_daily_df = spark.read.format("delta").load(gold_daily_summary_path)
gold_daily_df.sort(col("event_timestamp").desc()).show(10)

# --- 2. VALIDATE VOLATILITY & DOMINANCE ---
print("📊 Checking Volatility Metrics & Market Cap:")
gold_vol_df = spark.read.format("delta").load(gold_volatility_path)
gold_vol_df.sort(col("event_timestamp").desc()).show(10)

In [0]:
# KPI: Total Crypto Market Cap Trend (from Silver Global)
print("📈 Total Crypto Market Cap Trend:")
spark.read.format("delta").load(silver_global_path) \
    .select("event_timestamp", "total_market_cap") \
    .orderBy("event_timestamp") \
    .show(10)

In [0]:
# KPI: Top 5 Gainers (last 24 hours) 
print("🏆 Top 5 Gainers (Last 24 Hours):")
gold_daily_df.select("symbol", "daily_return") \
    .orderBy(col("daily_return").desc()) \
    .limit(5) \
    .show()

In [0]:
# Feature Validation: Price vs Moving Average (BTC Example)
print("🔍 BTC Price vs 7-Day Moving Average:")
gold_daily_df.filter(col("symbol") == "BTC") \
    .select("event_timestamp", "price_usd", "7d_moving_avg") \
    .orderBy(col("event_timestamp").desc()) \
    .show(10)